## Analyze Predictive Words

In [1]:
import pandas as pd

import sys
import os

sys.path.append(os.path.abspath("../src"))

In [2]:
from sentiment_model import SentimentModel
import torch.nn as nn
import torch
from tokenizer import Tokenizer



In [3]:
model = SentimentModel(vocab_size=10000,embedding_dim=100)
    
model.load_state_dict(torch.load("./../checkpoints/sentiment_model.pth",map_location='cpu'))
model.eval()

SentimentModel(
  (embedding): Embedding(10000, 100)
  (fc1): Linear(in_features=100, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=2, bias=True)
  (Relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
)

In [4]:
tokenizer = Tokenizer()

tokenizer.load()



In [5]:
word_influences = []

with torch.no_grad():
    for token_id,word in tokenizer.id_to_word.items():
       #1. Turn the ID into a tensor [[token_id]]
       input_tensor = torch.tensor([[token_id]]).long()
       
       #2. Get the model's output
       logits = model(input_tensor)
       
       #3. GEt teh probability of "Positive" (Index 1)
       probs = torch.softmax(logits, dim=1).squeeze()       
       #print(probs)
       pos_score =probs[1].item()
       
       
       word_influences.append((word,pos_score))
       
        
        
        

DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Input shape at start of forward: torch.Size([1, 1

In [6]:
#sort in descending order (Highest to lowest)
word_influences.sort(key=lambda x:x[1],reverse=True)

print(word_influences[:10])
print(word_influences[-20:])

[('you', 1.0), ('my', 1.0), ('well', 1.0), ('great', 1.0), ('also', 1.0), ('seen', 1.0), ('love', 1.0), ('best', 1.0), ('know', 1.0), ('still', 1.0)]
[('worse', 2.802596928649634e-45), ('unfunny', 2.802596928649634e-45), ('worst', 0.0), ('boring', 0.0), ('terrible', 0.0), ('awful', 0.0), ('waste', 0.0), ('annoying', 0.0), ('poorly', 0.0), ('lame', 0.0), ('fails', 0.0), ('disappointing', 0.0), ('laughable', 0.0), ('disappointment', 0.0), ('mediocre', 0.0), ('dreadful', 0.0), ('forgettable', 0.0), ('410', 0.0), ('310', 0.0), ('210', 0.0)]


In [7]:
#save words

def save_predictive_words():
    with open("./../experiments/predictive_words.txt","w",encoding="utf-8") as f:
        f.write(f"Positive words: \n")
        f.write(" \n")
        for word,prob in word_influences[:10]:
            f.write(f"{word}\n")
        
        f.write(" \n")
        f.write(" \n")

        f.write(f"Negative words: \n")
        f.write(" \n")
        for word,prob in word_influences[-10:]:
            f.write(f"{word}\n")

In [8]:
save_predictive_words()

## PREDICTING

In [10]:
import html

def visualize_attention(sentence, model, tokenizer, filename="experiments/attention_examples.html"):
    model.eval()
    words = sentence.split()
    highlights = []

    with torch.no_grad():
        for word in words:
            # Get the positive score for this specific word
            _, score = model.predict(word, tokenizer)
            # We normalize the score for the background-color (0.5 is neutral)
            # Higher than 0.5 = Green (Positive), Lower than 0.5 = Red (Negative)
            highlights.append((word, score))

    # Generate HTML string
    html_output = "<html><body style='font-family: sans-serif; padding: 20px;'>"
    html_output += f"<h3>Sentence: \"{sentence}\"</h3><div style='line-height: 2.5;'>"

    for word, score in highlights:
        # Map score to color intensity
        print(f"The score of the word is : {score}")
        if score > 0.5:
            # Green highlight for positive influence
            alpha = (score - 0.5) * 2  # Scale 0.5-1.0 to 0.0-1.0
            color = f"rgba(0, 255, 0, {alpha:.2f})"
        else:
            # Red highlight for negative influence
            alpha = (0.5 - score) * 2  # Scale 0.0-0.5 to 0.0-1.0
            color = f"rgba(255, 0, 0, {alpha:.2f})"
            
        html_output += f"<span style='background-color: {color}; padding: 5px; margin: 2px; border-radius: 3px;'>{html.escape(word)}</span> "

    html_output += "</div></body></html>"

    with open(filename, "w", encoding="utf-8") as f:
        f.write(html_output)
    print(f"✅ Visualization saved to {filename}")

In [11]:
visualize_attention("This movie was absolutely amazing",model,tokenizer,"./../experiments/attention_examples.html")

this is are sentences we are expecting :['This']
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Shape of probs is torch.Size([1, 2])
this is are sentences we are expecting :['movie']
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Shape of probs is torch.Size([1, 2])
this is are sentences we are expecting :['was']
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Shape of probs is torch.Size([1, 2])
this is are sentences we are expecting :['absolutely']
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Shape of probs is torch.Size([1, 2])
this is are sentences we are expecting :['amazing']
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Shape of probs is torch.Size([1, 2])
The score of the word is : 0.04861671105027199
The score of the word is : 0.4312817454338074
The score of the word is : 0.0022784804459661245
The score of the word is : 1.0
The score of the word is : 1.0
✅ Visualization saved to ./

In [11]:
words_to_test = ["amazing", "movie", "this","terrible","boring","bad"]
for w in words_to_test:
    label, confidence = model.predict(w, tokenizer)
    print(f"Word: {w:10} | Label: {label:10} | Confidence: {confidence:.4f}")

this is are sentences we are expecting :['amazing']
DEBUG: tokens content: [[489]]
DEBUG: tokens type: <class 'list'>
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Shape of probs is torch.Size([1, 2])
Word: amazing    | Label: Positive   | Confidence: 1.0000
this is are sentences we are expecting :['movie']
DEBUG: tokens content: [[18]]
DEBUG: tokens type: <class 'list'>
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Shape of probs is torch.Size([1, 2])
Word: movie      | Label: Negative   | Confidence: 0.5687
this is are sentences we are expecting :['this']
DEBUG: tokens content: [[11]]
DEBUG: tokens type: <class 'list'>
DEBUG: Input shape at start of forward: torch.Size([1, 1])
DEBUG: Shape of probs is torch.Size([1, 2])
Word: this       | Label: Negative   | Confidence: 0.9514
this is are sentences we are expecting :['terrible']
DEBUG: tokens content: [[378]]
DEBUG: tokens type: <class 'list'>
DEBUG: Input shape at start of forward: torch.Size(

### Full sentence prediction

In [ ]:
sentence = "This movie was absolutely amazing"

label, confidence = model.predict(sentence,tokenizer)

print(f"Sentence: {sentence}")
print(f"Label: {label} | Positive Probability: {confidence:.4f}")

this is are sentences we are expecting :['This movie was absolutely amazing']
DEBUG: Input shape at start of forward: torch.Size([1, 5])
DEBUG: Shape of probs is torch.Size([1, 2])
Sentence: This movie was absolutely amazing
Label: Positive | Positive Probability: 1.0000
